# 한국어 랩 생성 학습 — Qwen2.5-1.5B (Colab T4)

친구의 `adapters/training/run_training.py` 기준 SFT + GRPO 2단계 학습을 Colab T4(16GB)에서 실행.

- **베이스 모델**: `Qwen/Qwen2.5-1.5B` (친구 코드의 3B → 다운사이즈, 다른 하이퍼파라미터는 그대로)
- **학습 시간 예상**: SFT 1~2h, GRPO 4~8h (T4 기준)
- **실행 단위**: `run_training.py --stage {sft|sanity|grpo|eval}` 호출

## 실행 시나리오

**Case 1) 한 세션에서 SFT+GRPO 끝까지**: A → B → C → D 차례로.

**Case 2) SFT 끝내고 새 세션에서 GRPO 이어가기**:
- 첫 세션: A → B → B3에서 Drive 백업
- 새 세션: A를 처음부터 재실행 (A5에서 Drive의 SFT 어댑터 자동 복원) → C → D

## 런타임
**런타임 → 런타임 유형 변경 → 하드웨어 가속기 = T4 GPU**

---
# Section A — 매 세션마다 실행

세션이 끊겨 다시 들어왔을 때도 이 섹션을 처음부터 끝까지 다시 돌린다.

### A1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
assert torch.cuda.is_available(), 'CUDA 사용 불가 — 런타임 유형을 T4 GPU로'
print(f'device       : {torch.cuda.get_device_name(0)}')
print(f'bf16 support : {torch.cuda.is_bf16_supported()}  (T4는 False — fp16으로 자동 폴백)')

### A2. 의존성 설치

docstring 그대로. g2pk는 설치 실패해도 자모 분해 폴백이 있어서 학습엔 지장 없음.

In [ ]:
!pip install -q -U "transformers>=4.45" "accelerate>=0.34" "peft>=0.13" \
                    "trl>=0.12" "bitsandbytes>=0.43" "datasets>=2.20"
!pip install -q jamo pronouncing tqdm pandas
!pip install -q g2pk || echo 'g2pk 설치 실패 — 자모 분해 폴백 사용'

### A3. Google Drive 마운트

SFT/GRPO 어댑터를 Drive에 백업해두면 세션 끊겨도 안전.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/baseline_rapshit'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive backup path: {DRIVE_DIR}')

### A4. Upload project files

**Source: `adapters_fast/training/`** (속도 최적화 버전). 원본 `adapters/training/`도 그대로 남아 있으니, 원본으로 돌리려면 아래 명령의 `adapters_fast` → `adapters`로만 바꿔서 재생성.

**Expected zip layout** (A4 checks these paths):
```
project.zip
|-- run_training.py              (from adapters_fast/training/run_training.py)
|-- training/
|   |-- sft_qwen.py              (from adapters_fast/training/sft_qwen.py)
|   |-- grpo_qwen.py             (from adapters_fast/training/grpo_qwen.py)
|   `-- prepare_dataset.py       (from adapters_fast/training/prepare_dataset.py)
|-- rhyme_scoring/
|-- tests/                       (includes test_phonetics.py)
`-- tests_data/                  (merged_final_dataset_analyzed.csv, prepared_dataset.jsonl)
```

> Note: Windows `Compress-Archive` can store paths like `training\\sft_qwen.py`. Colab may extract that as a single filename instead of a directory path. The Python command below forces zip entries like `training/sft_qwen.py`.

**PowerShell (run at project root)**:
```powershell
$env:PYTHONIOENCODING = 'utf-8'
python -c "import zipfile, pathlib; files={'adapters_fast/training/run_training.py':'run_training.py','adapters_fast/training/sft_qwen.py':'training/sft_qwen.py','adapters_fast/training/grpo_qwen.py':'training/grpo_qwen.py','adapters_fast/training/prepare_dataset.py':'training/prepare_dataset.py'}; dirs=['rhyme_scoring','tests','tests_data']; z=zipfile.ZipFile('project.zip','w',zipfile.ZIP_DEFLATED); [z.write(src, arc) for src, arc in files.items()]; [z.write(str(p), p.as_posix()) for d in dirs for p in pathlib.Path(d).rglob('*') if p.is_file() and '__pycache__' not in p.parts]; z.close(); print('OK: project.zip created')"
```

**Bash/WSL alternative**:
```bash
python - <<'PY'
import zipfile, pathlib
files = {
    'adapters_fast/training/run_training.py': 'run_training.py',
    'adapters_fast/training/sft_qwen.py': 'training/sft_qwen.py',
    'adapters_fast/training/grpo_qwen.py': 'training/grpo_qwen.py',
    'adapters_fast/training/prepare_dataset.py': 'training/prepare_dataset.py',
}
with zipfile.ZipFile('project.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for src, arc in files.items():
        z.write(src, arc)
    for d in ['rhyme_scoring', 'tests', 'tests_data']:
        for p in pathlib.Path(d).rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts:
                z.write(p, p.as_posix())
print('OK: project.zip created')
PY
```

In [ ]:
import os
from google.colab import files

# Upload from a stable directory; the project directory may be deleted/recreated below.
os.chdir('/content')
print('project.zip upload...')
uploaded = files.upload()

In [ ]:
import os, zipfile, shutil
from pathlib import Path

PROJECT = '/content/baseline_rapshit'
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
os.makedirs(PROJECT, exist_ok=True)

for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall(PROJECT)
        print(f'Extracted: {name} -> {PROJECT}')

%cd {PROJECT}
!find . -maxdepth 2 -type f | sort | head -80

# Validate required files
required = [
    'run_training.py',
    'training/sft_qwen.py',
    'training/grpo_qwen.py',
    'training/prepare_dataset.py',
    'rhyme_scoring/rhyme_engine.py',
    'tests/test_phonetics.py',
    'tests_data/merged_final_dataset_analyzed.csv',
    'tests_data/prepared_dataset.jsonl',
]
missing = [p for p in required if not Path(p).exists()]
if missing:
    for p in missing: print(' missing:', p)
    raise FileNotFoundError('Required files are missing from project.zip')
print('\nOK: all required files found')

### A5. 이전 학습 어댑터 복원 (있으면)

Drive에 백업된 SFT/GRPO 어댑터가 있으면 로컬 `./models/`로 복원.  
**새 세션에서 GRPO 이어 가는 경우 핵심.**

In [ ]:
import os, shutil
os.makedirs('./models', exist_ok=True)

for name in ['sft_rap_qwen', 'grpo_rap_qwen']:
    drive_path = f'{DRIVE_DIR}/{name}'
    local_path = f'./models/{name}'
    if os.path.exists(drive_path) and not os.path.exists(local_path):
        print(f'복원: {drive_path} → {local_path}')
        shutil.copytree(drive_path, local_path)
    elif os.path.exists(local_path):
        print(f'이미 로컬에 존재: {local_path}')
    else:
        print(f'(Drive에 없음, 신규 학습) {drive_path}')

!ls -la ./models/ 2>/dev/null || echo 'models/ 비어있음 — 신규 학습'

### A6. MODEL_ID 패치 (3B → 1.5B)

친구 코드는 `./models/Qwen2.5-3B`(친구 서버 로컬 경로)를 가리킴. Colab에선 HF 허브 ID + 1.5B로 교체.

**모델만 바뀜. 다른 하이퍼파라미터 안 건드림.**

In [ ]:
import re
from pathlib import Path

NEW_MODEL = 'Qwen/Qwen2.5-1.5B'
targets = ['training/sft_qwen.py', 'training/grpo_qwen.py', 'run_training.py']
pattern = re.compile(r'MODEL_ID\s*=\s*["\'][^"\']+["\']')

for t in targets:
    p = Path(t)
    text = p.read_text(encoding='utf-8')
    new_text, n = pattern.subn(f'MODEL_ID = "{NEW_MODEL}"', text)
    if n:
        p.write_text(new_text, encoding='utf-8')
        print(f'[patched] {t} — {n}곳')
    else:
        print(f'[skip] {t} — MODEL_ID 못 찾음')

print('\n--- 패치 결과 ---')
!grep -nH MODEL_ID training/sft_qwen.py training/grpo_qwen.py run_training.py

### A7. Phonetics 회귀 테스트

한글/영어 음소화 모듈 sanity. 실패하면 학습 들어가지 말 것.

In [ ]:
!python tests/test_phonetics.py

---
# Section B — SFT 학습

Qwen2.5-1.5B + 4bit QLoRA. T4에서 1~2h 예상.  
어댑터 이미 있으면 자동 스킵 (재학습은 `--force`).  
끝나면 B2에서 Drive 백업 → 세션 끊겨도 안전.

### B1. SFT 실행

내부 동작:
- `prepared_dataset.jsonl` 없으면 자동 생성
- `./models/sft_rap_qwen` 없으면 학습, 있으면 스킵
- `./outputs/sft_qwen/checkpoint-*` 있으면 자동 재개

In [ ]:
import os, shutil
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# SFT 체크포인트를 Drive에 직접 쓰도록 symlink (세션 끊겨도 ./outputs/sft_qwen/checkpoint-* 살아남음)
drive_ckpt = f'{DRIVE_DIR}/sft_qwen_checkpoints'
local_ckpt = './outputs/sft_qwen'
os.makedirs('./outputs', exist_ok=True)
os.makedirs(drive_ckpt, exist_ok=True)

if os.path.lexists(local_ckpt) and not os.path.islink(local_ckpt):
    # 기존 로컬 체크포인트가 있으면 Drive로 옮긴 뒤 symlink로 교체
    if os.path.isdir(local_ckpt) and os.listdir(local_ckpt) and not os.listdir(drive_ckpt):
        shutil.copytree(local_ckpt, drive_ckpt, dirs_exist_ok=True)
    shutil.rmtree(local_ckpt)
if not os.path.lexists(local_ckpt):
    os.symlink(drive_ckpt, local_ckpt, target_is_directory=True)
print(f'SFT checkpoints -> {drive_ckpt}')

!python run_training.py --stage sft

### B2. SFT 어댑터 Drive 백업

In [ ]:
import os, shutil
src = './models/sft_rap_qwen'
dst = f'{DRIVE_DIR}/sft_rap_qwen'

assert os.path.exists(src), f'SFT 어댑터 없음: {src} — B1이 끝까지 돌았는지 확인'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'✓ 백업 완료: {dst}')
!ls -la {dst}

---
# Section C — GRPO 학습

SFT 어댑터 위에 라임 보상 RL. T4 + 1.5B에서 4~8h 예상.

> **Colab Free**에서는 GRPO 완주가 빠듯합니다. 12h max / idle 제한 때문에 C3 완료 전에 세션이 끊길 수 있어 가능하면 **Colab Pro 권장**.  
> **OOM 나면** `training/grpo_qwen.py`의 `max_completion_length=256`을 **192**로 먼저 낮춰 보고, 그래도 부족하면 `160`으로 재시도. `num_generations=4`는 GRPO 신호 품질에 직결되니 건드리지 말 것.

### C1. SFT 어댑터 가드

In [ ]:
import os
assert os.path.exists('./models/sft_rap_qwen'), \
    'SFT 어댑터 없음 — Section B 먼저 실행하거나 A5의 Drive 복원 확인'
!ls ./models/sft_rap_qwen | head -5

### C2. Reward sanity check

GRPO 들어가기 전 SFT 모델의 reward 분포 확인:
- `std < 0.05` → 학습 신호 부족 우려
- `mean < 0` → 페널티 과도, 가중치 조정 권장

In [ ]:
!python run_training.py --stage sanity

### C3. GRPO 학습 실행

save_steps=25 → 자주 체크포인트 저장, 끊겨도 `./outputs/grpo_qwen/checkpoint-*`에서 재개.

In [ ]:
import os, shutil
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

drive_ckpt = f'{DRIVE_DIR}/grpo_qwen_checkpoints'
local_ckpt = './outputs/grpo_qwen'
os.makedirs('./outputs', exist_ok=True)
os.makedirs(drive_ckpt, exist_ok=True)

# Write GRPO checkpoints directly to Drive so runtime resets can resume.
if os.path.lexists(local_ckpt) and not os.path.islink(local_ckpt):
    if os.path.isdir(local_ckpt) and os.listdir(local_ckpt) and not os.listdir(drive_ckpt):
        shutil.copytree(local_ckpt, drive_ckpt, dirs_exist_ok=True)
    shutil.rmtree(local_ckpt)
if not os.path.lexists(local_ckpt):
    os.symlink(drive_ckpt, local_ckpt, target_is_directory=True)
print(f'GRPO checkpoints -> {drive_ckpt}')

!python run_training.py --stage grpo

### C4. GRPO 어댑터 Drive 백업

In [ ]:
import os, shutil
src = './models/grpo_rap_qwen'
dst = f'{DRIVE_DIR}/grpo_rap_qwen'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'GRPO adapter backed up: {dst}')
else:
    print(f'[WARN] {src} not found; GRPO checkpoints are already written to Drive in C3')
    ck_dst = f'{DRIVE_DIR}/grpo_qwen_checkpoints'
    if os.path.exists(ck_dst):
        print(f'Checkpoint path: {ck_dst}')
    else:
        print('[WARN] No checkpoint found yet')

---
# Section D — 생성 평가

GRPO 어댑터(없으면 SFT)로 Tablo / Verbal Jint 프롬프트 샘플 출력.

In [ ]:
!python run_training.py --stage eval

---
## 트러블슈팅

### OOM
- **SFT 단계**: `training/sft_qwen.py` `per_device_train_batch_size=2` → `1`, `gradient_accumulation_steps=8` → `16` (effective 동일하게 유지)
- **GRPO 단계**: `training/grpo_qwen.py` `max_completion_length=256` → `192`로 먼저 낮추고, 여전히 OOM이면 `160`. **`num_generations=4`는 건드리지 말 것.**

### bitsandbytes 에러
Colab은 보통 깔려있지만 가끔 호환 문제 발생: `!pip install -U bitsandbytes` 후 런타임 재시작.

### 세션 끊김
- SFT는 `./outputs/sft_qwen/checkpoint-*`에서 자동 재개
- GRPO도 동일
- B2/C4 셀에서 Drive 백업해뒀으면 새 세션에서 zip 다시 풀고 A 섹션 처음부터 → A5가 자동 복원

### Reward std 너무 낮음 (< 0.05)
SFT가 균일한 출력을 내고 있다는 신호. 친구 코드 그대로 돌려서 발생하면 데이터 특성 문제일 가능성. reward 가중치 조정으로 대응.